# Solutions · 07 · Mixture fundamentals

Worked solutions to the exercises of [notebook 07](../notebooks/07_mixture_fundamentals.ipynb). Try each exercise
yourself before reading its solution - the learning happens in the attempt. Solutions are one way to
answer each question; other correct approaches exist.

In [1]:
try:                      # on Google Colab (or anywhere engthermo is missing): install it from GitHub
    import engthermo
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/ktwyw/engthermo"], check=True)
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from engthermo import datasets, mixtures
from engthermo.constants import R
plt.rcParams.update({"figure.figsize": (7, 4), "figure.dpi": 90, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

import inspect
from IPython.display import Code

def show_source(obj):
    """Display the source code of a library function or class."""
    return Code(inspect.getsource(obj), language="python")

def heading(text):
    print(f"\n{text}\n" + "=" * len(text))

In [2]:
d = pd.read_csv(datasets.path("ethanol_water_volume.csv"), comment="#")
x1, V = d.x_ethanol.to_numpy(), d.V_cm3_mol.to_numpy()

## Exercise 1

How much volume is 'lost' when 1 L of ethanol is mixed with 1 L of water at 25 degC? (Densities: ethanol 0.785 g/mL, water 0.997 g/mL; use the data file.)

In [3]:
M_e, M_w = 46.07, 18.02
n_e, n_w = 1000 * 0.785 / M_e, 1000 * 0.997 / M_w                   # moles in 1 L of each
x = n_e / (n_e + n_w)
V_mix = (n_e + n_w) * np.interp(x, x1, V)
V_pure = n_e * V[-1] + n_w * V[0]
print(f"x_ethanol = {x:.3f}; volume of the mixture {V_mix/1e3:.2f} L against {V_pure/1e3:.2f} L for the two liquids: {V_pure - V_mix:.0f} mL lost")

x_ethanol = 0.235; volume of the mixture 1.94 L against 2.00 L for the two liquids: 64 mL lost


About 3 % of the volume vanishes on mixing - a bartender's rule of thumb and a nuisance for anyone who
meters alcohol by volume. The molar volumes of the file give the answer through the same partial-molar
reasoning as the notebook.

## Exercise 2

The minimum work to separate 1 mol of an ideal equimolar mixture at 25 degC equals $-\Delta G_{mix}$. Compare it with the heat of vaporisation of benzene (about 31 kJ/mol) to see why distillation uses far more energy than the thermodynamic minimum.

In [4]:
W_min = R * 298.15 * np.log(2)                                     # -Delta G_mix per mole of equimolar mixture
print(f"minimum separation work {W_min/1e3:.2f} kJ/mol of mixture; heat of vaporisation of benzene about 31 kJ/mol -> ratio {31e3/W_min:.0f}")

minimum separation work 1.72 kJ/mol of mixture; heat of vaporisation of benzene about 31 kJ/mol -> ratio 18


Distillation boils the whole feed at least once - and often several times, in the reflux - so it spends
twenty or more times the thermodynamic minimum. That gap is the reason for heat integration, heat pumps on
columns, and the search for membrane and adsorption alternatives in the chemical industry.

## Exercise 3

**Implement it yourself:** fit the excess volume $V^E = x_1 x_2 (A + B(2x_1 - 1) + C(2x_1 - 1)^2)$ (Redlich-Kister) to the data with `np.linalg.lstsq`, and use it to compute the partial molar volumes at infinite dilution ($\bar V_1^\infty = V_1 + A - B + C$).

In [5]:
V_ideal = x1 * V[-1] + (1 - x1) * V[0]
VE = V - V_ideal
inner = (x1 > 0) & (x1 < 1)
X = np.column_stack([np.ones(inner.sum()), 2 * x1[inner] - 1, (2 * x1[inner] - 1) ** 2])
coef, *_ = np.linalg.lstsq(X, VE[inner] / (x1[inner] * (1 - x1[inner])), rcond=None)
A, B, C = coef
print(f"Redlich-Kister: A = {A:.3f}, B = {B:.3f}, C = {C:.3f} cm3/mol")
print(f"partial molar volume at infinite dilution: ethanol in water {V[-1] + A - B + C:.2f} cm3/mol (pure {V[-1]:.2f}), water in ethanol {V[0] + A + B + C:.2f} cm3/mol (pure {V[0]:.2f})")

Redlich-Kister: A = -4.350, B = 1.000, C = -0.299 cm3/mol
partial molar volume at infinite dilution: ethanol in water 53.03 cm3/mol (pure 58.68), water in ethanol 14.42 cm3/mol (pure 18.07)


The Redlich-Kister expansion turns the excess volume into three numbers, and its end-points give the
infinite-dilution partial molar volumes directly: an ethanol molecule alone among water molecules takes about
6 cm$^3$/mol less space than in pure ethanol, a water molecule alone among ethanol molecules about 4 cm$^3$/mol less.